# OCR → HTML no Colab (Ollama ou OpenRouter)

Processa várias notas de uma vez: envie as imagens juntas ou num `.zip`.

Com `ENGINE = "ollama"`, tudo roda no Colab sem custo: o `qwen2.5vl:7b` fica na GPU (use o ambiente T4).
Com `ENGINE = "openrouter"`, o `qwen/qwen3.8-27b` é pago e precisa do secret `OPENROUTER_API_KEY`.

In [ ]:
ENGINE = "ollama"  # ollama: qwen2.5vl:7b na gpu do colab, sem custo; openrouter: qwen3.8 pago, precisa do secret
MODEL = {"ollama": "qwen2.5vl:7b", "openrouter": "qwen/qwen3.8-27b"}[ENGINE]
INPUT = "input/amostra_diversa/captura/300"  # pasta ou arquivo; no colab, as notas enviadas vão para notas/
LIMIT = 0  # máximo de notas; 0 = todas
KINDS = ("compact",)  # entradas do llm: compact, markdown, full_json
VLM = True  # também gera o html direto da imagem, sem ocr, para comparar
OUTPUT = "output/notebook"
CPU_THREADS = 2
REASONING = False  # só openrouter: ligado, o qwen3.8 gasta 5–16 mil tokens pensando e pode truncar

In [ ]:
import os, sys
from pathlib import Path

if "google.colab" in sys.modules:
    ROOT = Path("/content")
    if not (ROOT / INPUT).exists():  # no colab, envie as notas: várias de uma vez ou um .zip
        import zipfile
        from google.colab import files

        INPUT, inbox = "notas", ROOT / "notas"
        inbox.mkdir(exist_ok=True)
        for name in files.upload():  # o colab salva os arquivos enviados em /content
            if name.lower().endswith(".zip"):
                with zipfile.ZipFile(ROOT / name) as archive:
                    archive.extractall(inbox, [m for m in archive.namelist() if not m.startswith("__MACOSX")])
                (ROOT / name).unlink()
            else:
                (ROOT / name).rename(inbox / name)
    if ENGINE == "openrouter":
        from google.colab import userdata
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    !pip install -q paddleocr==3.7.0 paddlepaddle==3.3.1 "paddlex[ocr]==3.7.2" openai python-dotenv beautifulsoup4 opencv-contrib-python
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")

from IPython.display import IFrame, display

Saving lote_diverso.zip to lote_diverso.zip


## Ollama

Só com `ENGINE = "ollama"`. No Colab a célula instala e sobe o servidor; o modelo (~6 GB)
é baixado de novo a cada sessão. Fora do Colab, rode antes `OLLAMA_CONTEXT_LENGTH=32768 ollama serve`.

In [ ]:
!pkill ollama || true
!rm -rf /usr/local/lib/ollama /usr/local/bin/ollama
!curl -fsSL https://ollama.com/install.sh | sh
!ls /usr/local/lib/ollama
!ollama --version

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
CPP_HTTPLIB_LICENSE	       libggml.so.0.27.0
cuda_v12		       libgomp.so.1
cuda_v13		       libgomp.so.1.0.0
DLPACK_LICENSE		       libllama-common.so
FMT_LICENSE		       libllama-common.so.0
GO_LICENSE		       libllama-common.so.0.7.0
libggml-base.so		       libllama-quantize-impl.so
libggml-base.so.0	       libllama-server-impl.so
libggml-base.so.0.27.0	       libllama.so
libggml-cpu-alderlake.so       libllama.so.0
libggml-cpu-cannonlake.so      libllama.so.0.7.0
libggml-cpu-cascadelake.so     libmtmd.so
libggml-cpu-cooperlake.so      libmtmd.so.0
libggml-cpu-haswell.so	       libmtmd.so.0.7.0


In [ ]:
if ENGINE == "ollama" and "google.colab" in sys.modules:
    import shutil, subprocess, time, urllib.request

    if not shutil.which("nvidia-smi"):
        print("Sem GPU: troque para T4 em Ambiente de execução → Alterar o tipo. Na CPU o qwen2.5vl leva minutos por nota.")
    if not shutil.which("ollama"):
        !apt-get -qq install -y zstd > /dev/null
        # Instala uma versão específica e estável do Ollama para evitar o erro do llama-server
        !curl -L https://ollama.com/download/ollama-linux-amd64.tgz -o ollama.tgz
        !tar -C /usr -xzf ollama.tgz
        !rm ollama.tgz
    # o contexto padrão do ollama corta o json da nota e a imagem sem avisar
    subprocess.Popen(["ollama", "serve"], env={**os.environ, "OLLAMA_CONTEXT_LENGTH": "32768"},
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(30):
        try:
            urllib.request.urlopen("http://localhost:11434")
            break
        except OSError:
            time.sleep(1)
    !ollama pull {MODEL}

## 0. Código da pipeline
Mesmo código do `pipeline.py`, que continua sendo usado pelo `main.py` local.

In [ ]:
# imports e caminhos
from contextlib import contextmanager
from datetime import datetime
from functools import cache
from html import escape
import json
import os
from pathlib import Path
import re
import threading
import time

from bs4 import BeautifulSoup
import psutil

BASE_DIR = ROOT  # definido na célula de preparo
OUTPUT_DIR = BASE_DIR / "output"
RUNS_DIR = BASE_DIR / "runs"
SUPPORTED_INPUTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".pdf"}
LLM_URLS = {"openrouter": "https://openrouter.ai/api/v1", "ollama": "http://localhost:11434/v1"}
MODELS = {"openrouter": "qwen/qwen3.8-27b",  # pago; a versão :free saiu da openrouter
          "ollama": "qwen2.5vl:7b"}  # roda na gpu do colab, sem custo
VLM_MAX_SIDE = 1600  # o qwen2.5-vl gasta ~1 token por 28×28 px; a captura de 300 dpi estoura o contexto
HTML_SUFFIX = {"compact": "_html.html", "markdown": "_from_md_html.html", "full_json": "_from_full_json_html.html"}

In [ ]:
# métricas

records, _active = [], []


@contextmanager
def stage(name):
    """mede tempo, cpu, pico de ram e tokens de llm de uma etapa."""
    record = {"stage": name, "peak_ram_mb": 0, "tokens_in": 0, "tokens_out": 0, "cost_usd": 0.0}
    stop, process = threading.Event(), psutil.Process()

    def sample():  # ru_maxrss marcou 7,9 GB onde a amostragem do rss viu 1,9 GB
        while True:
            record["peak_ram_mb"] = max(record["peak_ram_mb"], process.memory_info().rss / 2**20)
            if stop.wait(.05):
                return

    sampler = threading.Thread(target=sample, daemon=True)
    sampler.start()
    _active.append(record)
    cpu, started = time.process_time(), time.perf_counter()
    try:
        yield record
    finally:
        _active.pop()
        stop.set()
        sampler.join()
        record.update(seconds=time.perf_counter() - started, cpu_seconds=time.process_time() - cpu)
        records.append(record)


def record_llm(tokens_in, tokens_out, cost=0.0):
    if _active:
        _active[-1]["tokens_in"] += tokens_in or 0
        _active[-1]["tokens_out"] += tokens_out or 0
        _active[-1]["cost_usd"] += float(cost or 0)


def summary():
    import pandas as pd

    table = pd.DataFrame(records).groupby("stage", sort=False).agg(
        calls=("seconds", "size"), seconds=("seconds", "sum"), cpu_seconds=("cpu_seconds", "sum"),
        peak_ram_mb=("peak_ram_mb", "max"), tokens_in=("tokens_in", "sum"), tokens_out=("tokens_out", "sum"),
        cost_usd=("cost_usd", "sum"))
    table.insert(2, "cpu_%", 100 * table.pop("cpu_seconds") / table.seconds)
    return table


def report(label="pipeline"):
    """imprime o resumo, salva em runs/metrics_<timestamp>.json e zera as medições."""
    if not records:
        return None
    table = summary()
    print(table.round(4).to_string())
    RUNS_DIR.mkdir(exist_ok=True)
    path = RUNS_DIR / f"metrics_{datetime.now():%Y%m%d_%H%M%S_%f}.json"
    path.write_text(json.dumps({"label": label, "cpu_count": os.cpu_count(), "stages": records,
                                "summary": table.reset_index().to_dict("records")},
                               indent=2, default=lambda value: value.item()), encoding="utf-8")
    records.clear()
    print(f"Métricas: {path}")
    return path

In [ ]:
# ocr


def extract_zip(archive_path, destination):
    """extrai um zip de notas (ignora o __MACOSX do mac) e devolve a pasta extraída."""
    import zipfile

    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(destination, [m for m in archive.namelist() if not m.startswith("__MACOSX")])
    return destination


def result_directories(results_dir):
    """separa intermediários e htmls finais em output/intermediarios e output/html."""
    root = Path(results_dir).resolve()
    if not root.is_relative_to(OUTPUT_DIR):
        return root / "intermediarios", root / "html"
    relative = root.relative_to(OUTPUT_DIR)
    if relative.parts and relative.parts[0] in {"html", "intermediarios"}:
        relative = Path(*relative.parts[1:])
    return OUTPUT_DIR / "intermediarios" / relative, OUTPUT_DIR / "html" / relative


def input_order(path, root):
    """prioriza imagens com maior resolução."""
    first = path.relative_to(root).parts[0]
    return -int(first) if first in {"100", "150", "200", "300"} else 0, str(path.relative_to(root))


def select_inputs(root, dpi=None, limit=0):
    files = sorted((path for path in root.rglob("*") if path.is_file() and path.suffix.lower() in SUPPORTED_INPUTS
                    and (not dpi or path.relative_to(root).parts[0] == str(dpi))), key=lambda p: input_order(p, root))
    return files[:limit] if limit else files


# textline_orientation=False: ligado, gira por engano linhas das notas em paisagem e o ocr lê lixo
# (campos 66,7% → 88,9% na nota 02-1-cinza-300, mesmo tempo)
def load_engine(lang="pt", doc_orientation=True, textline_orientation=False, det_max_side=1280, cpu_threads=2):
    from paddleocr import PPStructureV3

    return PPStructureV3(
        lang=None if lang == "default" else lang, ocr_version=None if lang == "default" else "PP-OCRv5",
        use_doc_orientation_classify=doc_orientation, use_textline_orientation=textline_orientation,
        use_doc_unwarping=False,  # capturas sintéticas já são planas
        enable_mkldnn=False,  # onednn falha no paddle 3.3.1 (ConvertPirAttribute2RuntimeAttribute)
        cpu_threads=cpu_threads,  # 2 foi o mais rápido no i7-1355U
        use_formula_recognition=False, use_chart_recognition=False, use_seal_recognition=False,
        use_table_recognition=True, text_det_limit_side_len=det_max_side, text_det_limit_type="max")


def simplify_result(res):
    """json compacto: só tipo, posição e conteúdo de cada bloco."""
    data = json.loads(res.json) if isinstance(res.json, str) else res.json
    return [{"id": i, "type": block.get("block_label"), "bbox": block.get("block_bbox"),
             "content": block.get("block_content")}
            for i, block in enumerate(data.get("res", data).get("parsing_res_list", []))]


def prepare_image(source, destination, min_width=0):
    """amplia até min_width; imagem já larga volta como está."""
    from PIL import Image

    with Image.open(source) as image:
        if image.width >= min_width:
            return source
        destination.parent.mkdir(parents=True, exist_ok=True)
        image.resize((min_width, round(image.height * min_width / image.width)),
                     Image.Resampling.LANCZOS).convert("RGB").save(destination)
        return destination


def process_file(engine, input_path, input_root, output, min_width=1600, ruled_tables=True):
    """roda o ocr em uma imagem ou pdf, salva as saídas e retorna (pasta, status)."""
    relative = input_path.relative_to(input_root)
    folder = result_directories(output)[0] / relative.parent / f"{input_path.stem}_{input_path.suffix[1:].lower()}"
    folder.mkdir(parents=True, exist_ok=True)
    status, started = "ok", time.perf_counter()
    try:
        source = input_path
        if input_path.suffix.lower() != ".pdf":
            with stage("ocr:preparar_imagem"):
                source = prepare_image(input_path, folder / "_inputs" / f"{input_path.stem}.png", min_width)
        with stage("ocr:predict"):
            pages = list(engine.predict(str(source)))
        if not pages:
            raise ValueError("OCR não retornou páginas")
        for page, res in enumerate(pages, start=1):
            if ruled_tables:
                with stage("ocr:reparar_tabelas"):
                    repair_tables(res, source)
            with stage("ocr:salvar"):
                res.save_to_json(save_path=str(folder))
                res.save_to_markdown(save_path=str(folder))
                res.save_to_html(save_path=str(folder))
                (folder / f"{input_path.stem}_page_{page:03d}_layout.json").write_text(
                    json.dumps(simplify_result(res), ensure_ascii=False, indent=2), encoding="utf-8")
    except Exception as exc:
        status = "failed"
        print(f"Falha em {input_path.name}: {exc}", flush=True)
    (folder / "ocr_run.json").write_text(json.dumps({"status": status, "seconds": round(time.perf_counter() - started, 3)}),
                                         encoding="utf-8")
    return folder, status

In [ ]:
# tabelas pelas bordas visíveis


def _boundaries(values, tolerance):
    """agrupa coordenadas próximas e devolve a média de cada grupo."""
    groups = []
    for value in sorted(values):
        if groups and value - groups[-1][-1] <= tolerance:
            groups[-1].append(value)
        else:
            groups.append([value])
    return [round(sum(group) / len(group)) for group in groups]


def _overlap(box, cell):
    x1, y1, x2, y2 = box
    a, b, c, d = cell
    return max(0, min(x2, c) - max(x1, a)) * max(0, min(y2, d) - max(y1, b)) / max(1, (x2-x1)*(y2-y1))


def detect_cells(image):
    import cv2
    import numpy as np

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.ndim == 3 else image
    _, ink = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    height, width = gray.shape
    horizontal = cv2.morphologyEx(ink, cv2.MORPH_OPEN, np.ones((1, max(20, width // 40)), np.uint8))
    vertical = cv2.morphologyEx(ink, cv2.MORPH_OPEN, np.ones((max(20, height // 60), 1), np.uint8))
    borders = cv2.morphologyEx(horizontal | vertical, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))
    _, _, stats, _ = cv2.connectedComponentsWithStats(255 - borders)
    return [[int(x), int(y), int(x+w), int(y+h)] for x, y, w, h, area in stats[1:]
            if w >= 12 and h >= 8 and area / (w*h) > .8 and x > 0 and y > 0 and x+w < width and y+h < height]


def ruled_table(image, ocr, bbox, cells=None):
    """reconstrói a tabela pela grade visível; sem bordas confiáveis, retorna None."""
    cells = [cell for cell in (detect_cells(image) if cells is None else cells) if _overlap(cell, bbox) > .9]
    if len(cells) < 8:
        return None
    eligible = [(i, text, box) for i, (text, box) in enumerate(zip(ocr.get("rec_texts", []), ocr.get("rec_boxes", [])))
                if text.strip() and _overlap(box, bbox) > .5]
    if not eligible:
        return None
    assigned, loose = [[] for _ in cells], []
    for i, text, box in eligible:
        hits = sorted([(_overlap(box, cell), j) for j, cell in enumerate(cells)], reverse=True)
        if hits[0][0] >= .55 and (len(hits) == 1 or hits[1][0] < .25):
            assigned[hits[0][1]].append((box[1], box[0], i, text))
        else:
            loose.append((box[1], i, text))
    matched = len(eligible) - len(loose)
    if matched / len(eligible) < .65:
        return None

    tolerance = max(3, round(image.shape[1] / 700))
    xs = _boundaries([v for c in cells for v in (c[0], c[2])], tolerance)
    ys = _boundaries([v for c in cells for v in (c[1], c[3])], tolerance)
    nearest = lambda value, edges: min(range(len(edges)), key=lambda i: abs(edges[i] - value))
    grid = [(nearest(c[0], xs), nearest(c[1], ys), nearest(c[2], xs), nearest(c[3], ys)) for c in cells]
    if any(left >= right or top >= bottom for left, top, right, bottom in grid):
        return None
    # faixas horizontais que nenhuma célula atravessa; faixas vizinhas com as mesmas colunas viram uma tabela
    cuts = [y for y in range(len(ys)) if not any(top < y < bottom for _, top, _, bottom in grid)]
    bands = []
    for lo, hi in zip(cuts, cuts[1:]):
        members = [i for i, (_, top, _, bottom) in enumerate(grid) if lo <= top < bottom <= hi]
        if not members:
            continue
        signature = tuple(sorted({v for i in members for v in (grid[i][0], grid[i][2])}))
        if bands and bands[-1]["end"] == lo and bands[-1]["signature"] == signature:
            bands[-1].update(end=hi, members=bands[-1]["members"] + members)
        else:
            bands.append({"start": lo, "end": hi, "signature": signature, "members": members})

    fragments = []
    for band in bands:
        members = band["members"]
        if not any(assigned[i] for i in members):
            continue
        columns = list(band["signature"])
        rows = sorted({v for i in members for v in (grid[i][1], grid[i][3])})
        html = ["<table><tbody>"]
        for top in rows[:-1]:
            html.append("<tr>")
            occupied = {x for i in members if grid[i][1] < top < grid[i][3]
                        for x in range(columns.index(grid[i][0]), columns.index(grid[i][2]))}
            starts = {columns.index(grid[i][0]): i for i in members if grid[i][1] == top}
            x = 0
            while x < len(columns)-1:
                if x in occupied or x not in starts:
                    html.append("" if x in occupied else "<td></td>")
                    x += 1
                    continue
                i = starts[x]
                left, _, right, bottom = grid[i]
                colspan, rowspan = columns.index(right) - columns.index(left), rows.index(bottom) - rows.index(top)
                attrs = (f' colspan="{colspan}"' if colspan > 1 else "") + (f' rowspan="{rowspan}"' if rowspan > 1 else "")
                html.append(f"<td{attrs}>{'<br>'.join(escape(line[3]) for line in sorted(assigned[i]))}</td>")
                x += colspan
            html.append("</tr>")
        html.append("</tbody></table>")
        fragments.append((ys[band["start"]], "".join(html)))
    fragments.extend((y, f"<p>{escape(text)}</p>") for y, _, text in loose)  # textos sem posição segura
    return {"html": "\n".join(fragment for _, fragment in sorted(fragments, key=lambda part: part[0])),
            "cells": cells, "matched_lines": matched, "loose_line_ids": [i for _, i, _ in loose]}


def repair_tables(res, input_path):
    blocks = [block for block in res["parsing_res_list"] if block.label == "table"]
    if not blocks:
        return
    import cv2

    preprocessed = res.get("doc_preprocessor_res", {})
    image = preprocessed.get("output_img")
    if image is None and not preprocessed.get("angle", 0):
        image = cv2.imread(str(input_path))
    if image is None:
        return
    cells = detect_cells(image)
    for block, table in zip(blocks, res["table_res_list"]):
        if repaired := ruled_table(image, res["overall_ocr_res"], block.bbox, cells):
            block.content = table["pred_html"] = repaired["html"]
            table["cell_box_list"] = repaired["cells"]

In [ ]:
# html

BASE_PROMPT = """Return only a complete HTML document with closed tags, no Markdown.
Preserve all recognized text exactly and in order; never correct, omit or invent content.
Keep labels with values and the relative layout of fields and sections.
Preserve every table, row, td/th, rowspan and colspan; do not merge unrelated cells.
"""

HTML_STYLE = """<style>
@page { size: A4 landscape; margin: 10mm; }
* { box-sizing: border-box; }
html { background: #eef1f5; }
body { margin: 24px auto; max-width: 1440px; padding: 24px;
       background: #fff; color: #202936; font: 14px/1.5 Arial, sans-serif;
       box-shadow: 0 4px 24px #18263914; }
.invoice-block + .invoice-block { margin-top: 20px; }
.table-scroll { overflow-x: auto; max-width: 100%; }
table { width: 100%; border-collapse: collapse; table-layout: auto; text-align: left; }
td, th { border: 1px solid #c4ccd5; padding: 8px 10px; vertical-align: top;
         overflow-wrap: break-word; min-width: 48px; }
th { background: #e8edf3; font-weight: 700; }
tr:nth-child(even) > td { background: #f7f9fc; }
td.numeric, th.numeric { white-space: nowrap; text-align: right; font-variant-numeric: tabular-nums; }
tr { break-inside: avoid; }
p { margin: 0 0 12px; white-space: pre-wrap; }
img { max-width: 100%; height: auto; }
@media (max-width: 600px) { body { margin: 0; padding: 12px; } }
@media print {
  html, body { background: #fff; }
  body { margin: 0; padding: 0; max-width: none; font-size: 8pt; box-shadow: none; }
  .table-scroll { overflow: visible; }
  td, th { padding: 3px; min-width: 0; }
  td.numeric, th.numeric { white-space: normal; }
}
</style>"""


def source_path(layout_path, kind):
    stem = layout_path.name.rsplit("_page_", 1)[0]
    return {"compact": layout_path, "markdown": layout_path.with_name(f"{stem}.md"),
            "full_json": layout_path.with_name(f"{stem}_res.json")}[kind]


def output_path(layout_path, kind, results_dir):
    intermediates_dir, html_dir = result_directories(results_dir)
    name = layout_path.name.removesuffix("_layout.json") + HTML_SUFFIX[kind]
    return (html_dir / layout_path.relative_to(intermediates_dir)).with_name(name)


def prepare_content(kind, path):
    content = path.read_text(encoding="utf-8")
    if kind == "markdown":
        return content
    data = json.loads(content)
    if kind == "full_json":
        data = [{"type": block.get("block_label"), "bbox": block.get("block_bbox"), "content": block.get("block_content")}
                for block in data.get("res", data).get("parsing_res_list", [])]
    return json.dumps(data, ensure_ascii=False, separators=(",", ":"))


def clean_fragment(content, keep_classes=False):
    """remove estilos e documentos aninhados; preserva células e mesclagens."""
    soup = BeautifulSoup(content, "html.parser")
    for tag in soup.find_all(["head", "style", "script", "meta", "link"]):
        tag.decompose()
    for tag in soup.find_all(["html", "body"]):
        tag.unwrap()
    allowed = {"rowspan", "colspan", "class"} if keep_classes else {"rowspan", "colspan"}
    for tag in soup.find_all(True):
        tag.attrs = {key: value for key, value in tag.attrs.items() if key in allowed}
    return soup


def render_source(kind, content):
    """html direto do ocr, sem llm."""
    fragments = [content] if kind == "markdown" else [
        str(block.get("content") or "") if block.get("type") == "table"
        else f"<p>{escape(str(block.get('content') or ''))}</p>" for block in json.loads(content)]
    soup = clean_fragment("\n".join(f"<section>{part}</section>" for part in fragments))
    for section in soup.find_all("section", recursive=False):
        section["class"] = "invoice-block"
    return add_presentation(soup)


def add_presentation(html):
    """aplica o estilo sem alterar o conteúdo das células."""
    soup = html if not isinstance(html, str) else clean_fragment(
        re.sub(r"^```(?:html)?\s*|\s*```$", "", html.strip(), flags=re.IGNORECASE), keep_classes=True)
    for cell in soup.find_all(["td", "th"]):
        if re.fullmatch(r"[+−-]?[\d.,%/ -]+", cell.get_text(strip=True)):
            cell["class"] = ["numeric"]
    for table in soup.find_all("table"):
        table.wrap(soup.new_tag("div", attrs={"class": "table-scroll", "tabindex": "0", "role": "region",
                                              "aria-label": "Tabela da nota fiscal"}))
    return ('<!doctype html><html lang="pt-BR"><head><meta charset="utf-8">'
            '<meta name="viewport" content="width=device-width, initial-scale=1">'
            f'<title>Nota fiscal</title>{HTML_STYLE}</head><body>{soup}</body></html>')


@cache
def _client(engine="openrouter"):
    from dotenv import load_dotenv
    from openai import OpenAI

    load_dotenv(BASE_DIR / ".env")
    key = os.environ["OPENROUTER_API_KEY"] if engine == "openrouter" else "ollama"  # o ollama ignora a chave
    return OpenAI(base_url=LLM_URLS[engine], api_key=key, max_retries=5)


def _ask(engine, model, content, num_predict=16384, reasoning=False):
    """uma chamada de chat; content é texto ou lista de partes (texto e imagem)."""
    response = _client(engine).chat.completions.create(
        model=model, temperature=0, max_tokens=num_predict, messages=[{"role": "user", "content": content}],
        # raciocínio gasta 5–16 mil tokens e trunca; o ollama não conhece o campo
        extra_body={"reasoning": {"enabled": reasoning}} if engine == "openrouter" else None)
    record_llm(response.usage.prompt_tokens, response.usage.completion_tokens, getattr(response.usage, "cost", 0))
    choice = response.choices[0]
    if choice.finish_reason == "length":
        raise ValueError("HTML truncado: aumente num-predict.")
    if not choice.message.content:
        raise ValueError("O modelo não retornou conteúdo.")
    return choice.message.content


def ask_llm(kind, content, model=None, num_predict=8192, reasoning=False, engine="openrouter"):
    source = ("Input: PaddleOCR Markdown with HTML tables." if kind == "markdown" else
              "Input: OCR JSON blocks. Use content as text/table markup, type as label, bbox for layout only; hide coordinates.")
    return _ask(engine, model or MODELS[engine], f"{BASE_PROMPT}{source}\n\n{content}", num_predict, reasoning)


def ask_vlm(image_path, model=None, num_predict=16384, engine="openrouter"):
    """html direto da imagem, sem ocr; a imagem vai reduzida a VLM_MAX_SIDE e em png."""
    import base64
    import cv2

    image = cv2.imread(str(image_path))
    if image is None:
        raise ValueError(f"Imagem ilegível (pdf não é aceito pelo vlm): {image_path}")
    if (scale := VLM_MAX_SIDE / max(image.shape[:2])) < 1:
        image = cv2.resize(image, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    data = base64.b64encode(cv2.imencode(".png", image)[1]).decode()
    return _ask(engine, model or MODELS[engine], [
        {"type": "text", "text": f"{BASE_PROMPT}Input: image of a Brazilian invoice (DANFE). Transcribe every printed text exactly."},
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{data}"}}], num_predict)


def write_html(layout_path, kind, results_dir, engine="direct", model=None, num_predict=8192, reasoning=False):
    """gera um html final (engine direct, openrouter ou ollama); o log fica nos intermediários."""
    source = source_path(layout_path, kind)
    if not source.is_file():
        print(f"Entrada {kind} ausente: {source}")
        return None
    destination = output_path(layout_path, kind, results_dir)
    destination.parent.mkdir(parents=True, exist_ok=True)
    log = {"source": str(source), "engine": engine, "status": "failed"}
    with stage(f"html:{engine}:{kind}") as record:
        try:
            content = prepare_content(kind, source)
            html = (render_source(kind, content) if engine == "direct" else
                    add_presentation(ask_llm(kind, content, model, num_predict, reasoning, engine)))
            destination.write_text(html, encoding="utf-8")
            if engine != "direct":
                log.update(model=model or MODELS[engine], **{k: record[k] for k in ("tokens_in", "tokens_out", "cost_usd")})
            log["status"] = "accepted"
            print(f"HTML ({kind}): {destination}", flush=True)
        except Exception as exc:
            log["error"] = str(exc)
            print(f"Falha no fluxo {kind} para {source.name}: {exc}", flush=True)
    log["duration_seconds"] = round(record["seconds"], 3)
    layout_path.with_name(destination.name).with_suffix(".generation.json").write_text(
        json.dumps(log, ensure_ascii=False, indent=2), encoding="utf-8")
    return log

In [ ]:
root, output = ROOT / INPUT, ROOT / OUTPUT
if root.suffix.lower() == ".zip":  # zip de notas: extrai e processa a pasta
    root = extract_zip(root, ROOT / "entradas" / root.stem)
input_root = root.parent if root.is_file() else root
images = [root] if root.is_file() else select_inputs(root, limit=LIMIT)
print(len(images), "notas:", ", ".join(str(p.relative_to(input_root)) for p in images))
show = lambda prefix: display(summary().filter(like=prefix, axis=0).round(2))

19 notas: captura/100/01-1-padrao-100/p01.png, captura/100/02-2a5-courier-100/p01.png, captura/100/03-6a15-cinza-100/p01.png, captura/100/04-1-espacada-100/p01.png, captura/100/17-16a40-padrao-100/p01.png, captura/100/18-2a5-courier-100/p01.png, captura/100/19-1-cinza-100/p01.png, captura/150/05-1-padrao-150/p01.png, captura/150/06-1-courier-150/p01.png, captura/150/07-1-cinza-150/p01.png, captura/150/08-1-espacada-150/p01.png, captura/200/09-1-padrao-200/p01.png, captura/200/10-2a5-courier-200/p01.png, captura/200/11-1-cinza-200/p01.png, captura/200/12-1-espacada-200/p01.png, captura/300/13-1-padrao-300/p01.png, captura/300/14-1-courier-300/p01.png, captura/300/15-1-cinza-300/p01.png, captura/300/16-1-espacada-300/p01.png


## 1. OCR

No Colab, o OCR leva ~2 min por nota na CPU. Uma nota que falha não para as outras.

In [ ]:
with stage("ocr:carregar_modelo"):
    engine = load_engine(cpu_threads=CPU_THREADS)
notes = []
for i, image in enumerate(images, 1):
    print(f"[{i}/{len(images)}] {image.name}", flush=True)
    folder, status = process_file(engine, image, input_root, output)
    notes.append({"image": image, "folder": folder, "status": status, "layouts": sorted(folder.glob("*_layout.json"))})
print(sum(note["status"] == "ok" for note in notes), "de", len(notes), "notas com OCR ok")
show("ocr")

/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Creating model: ('PP-DocBlockLayout', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-DocBlockLayout`.
Creating model: ('PP-DocLayout_plus-L', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-DocLayout_plus-L`.
Creating model: ('PP-OCRv5_server_det', None, None)
Model f

[1/19] p01.png


Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv5_server_det`.
Creating model: ('latin_PP-OCRv5_mobile_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/latin_PP-OCRv5_mobile_rec`.


[2/19] p01.png
[3/19] p01.png
[4/19] p01.png


/tmp/ipykernel_69584/2288901256.py:18: RuntimeWarning: overflow encountered in scalar multiply
  return max(0, min(x2, c) - max(x1, a)) * max(0, min(y2, d) - max(y1, b)) / max(1, (x2-x1)*(y2-y1))


[5/19] p01.png
[6/19] p01.png
[7/19] p01.png
[8/19] p01.png
[9/19] p01.png
[10/19] p01.png
[11/19] p01.png
[12/19] p01.png
[13/19] p01.png
[14/19] p01.png
[15/19] p01.png
[16/19] p01.png
[17/19] p01.png


/tmp/ipykernel_69584/2288901256.py:18: RuntimeWarning: overflow encountered in scalar multiply
  return max(0, min(x2, c) - max(x1, a)) * max(0, min(y2, d) - max(y1, b)) / max(1, (x2-x1)*(y2-y1))


[18/19] p01.png
[19/19] p01.png
19 de 19 notas com OCR ok


,calls,seconds,cpu_%,peak_ram_mb,tokens_in,tokens_out,cost_usd
stage,,,,,,,
ocr:carregar_modelo,1,29.46,60.60,2225.25,0,0,0.0
ocr:preparar_imagem,19,6.73,103.54,2702.06,0,0,0.0
ocr:predict,19,1361.83,113.20,3004.19,0,0,0.0
ocr:reparar_tabelas,19,1.22,144.21,2749.03,0,0,0.0
ocr:salvar,19,2.59,108.47,2749.03,0,0,0.0


## 2. HTML via LLM (OCR → LLM)

In [ ]:
for note in notes:  # um html por página e por entrada do llm
    note["logs"] = {kind: [write_html(layout, kind, output, engine=ENGINE, model=MODEL, reasoning=REASONING)
                           for layout in note["layouts"]] for kind in KINDS}
show("html")

HTML (compact): /content/output/html/notebook/captura/100/01-1-padrao-100/p01_png/p01_page_001_html.html
HTML (compact): /content/output/html/notebook/captura/100/02-2a5-courier-100/p01_png/p01_page_001_html.html
HTML (compact): /content/output/html/notebook/captura/100/03-6a15-cinza-100/p01_png/p01_page_001_html.html
HTML (compact): /content/output/html/notebook/captura/100/04-1-espacada-100/p01_png/p01_page_001_html.html
HTML (compact): /content/output/html/notebook/captura/100/17-16a40-padrao-100/p01_png/p01_page_001_html.html
HTML (compact): /content/output/html/notebook/captura/100/18-2a5-courier-100/p01_png/p01_page_001_html.html
HTML (compact): /content/output/html/notebook/captura/100/19-1-cinza-100/p01_png/p01_page_001_html.html
HTML (compact): /content/output/html/notebook/captura/150/05-1-padrao-150/p01_png/p01_page_001_html.html
HTML (compact): /content/output/html/notebook/captura/150/06-1-courier-150/p01_png/p01_page_001_html.html
HTML (compact): /content/output/html/note

,calls,seconds,cpu_%,peak_ram_mb,tokens_in,tokens_out,cost_usd
stage,,,,,,,
html:ollama:compact,19,1225.48,0.39,2682.73,31519,37598,0.0


In [ ]:
first = next((note for note in notes if note["layouts"]), None)  # mostra a primeira; as outras vão no zip
IFrame(os.path.relpath(output_path(first["layouts"][0], KINDS[0], output)), width="100%", height=600) if first else None

## 3. HTML direto da imagem (VLM, sem OCR)

O mesmo modelo lê a imagem, reduzida a `VLM_MAX_SIDE` px. PDF não é aceito aqui, e as notas sem OCR são puladas.

In [ ]:
for note in notes:
    note["vlm"] = None
    if not VLM or not note["layouts"] or note["image"].suffix.lower() == ".pdf":
        continue
    path = output_path(note["layouts"][0], "compact", output).with_name(f"{note['image'].stem}_vlm.html")
    with stage(f"vlm:{ENGINE}") as record:
        try:
            path.write_text(add_presentation(ask_vlm(note["image"], MODEL, engine=ENGINE)), encoding="utf-8")
            note["vlm"] = path, record
            print(f"HTML (vlm): {path}", flush=True)
        except Exception as exc:
            print(f"Falha no vlm em {note['image'].name}: {exc}", flush=True)
show("vlm")

HTML (vlm): /content/output/html/notebook/captura/100/01-1-padrao-100/p01_png/p01_vlm.html
Falha no vlm em p01.png: HTML truncado: aumente num-predict.
HTML (vlm): /content/output/html/notebook/captura/100/03-6a15-cinza-100/p01_png/p01_vlm.html
HTML (vlm): /content/output/html/notebook/captura/100/04-1-espacada-100/p01_png/p01_vlm.html
Falha no vlm em p01.png: HTML truncado: aumente num-predict.
HTML (vlm): /content/output/html/notebook/captura/100/18-2a5-courier-100/p01_png/p01_vlm.html
HTML (vlm): /content/output/html/notebook/captura/100/19-1-cinza-100/p01_png/p01_vlm.html
HTML (vlm): /content/output/html/notebook/captura/150/05-1-padrao-150/p01_png/p01_vlm.html
Falha no vlm em p01.png: HTML truncado: aumente num-predict.
Falha no vlm em p01.png: HTML truncado: aumente num-predict.
HTML (vlm): /content/output/html/notebook/captura/150/08-1-espacada-150/p01_png/p01_vlm.html
HTML (vlm): /content/output/html/notebook/captura/200/09-1-padrao-200/p01_png/p01_vlm.html
HTML (vlm): /content

In [ ]:
first_vlm = next((note["vlm"][0] for note in notes if note["vlm"]), None)
IFrame(os.path.relpath(first_vlm), width="100%", height=600) if first_vlm else None

## 4. Tempo por caminho

O OCR não conta o carregamento do modelo; no Colab, o primeiro `predict` inclui o aquecimento.

In [ ]:
import pandas as pd

rows = []
for note in notes:
    ocr_seconds = json.loads((note["folder"] / "ocr_run.json").read_text())["seconds"]
    row = {"nota": str(note["image"].relative_to(input_root)), "OCR": note["status"]}
    if note["status"] == "ok":
        row["OCR → HTML direto (sem LLM)"] = ocr_seconds
    for kind, logs in note["logs"].items():  # pdf: soma as páginas
        if logs and all(log and log["status"] == "accepted" for log in logs):
            row[f"OCR → {ENGINE} ({kind})"] = ocr_seconds + sum(log["duration_seconds"] for log in logs)
    if note["vlm"]:
        row[f"VLM {ENGINE} (imagem)"] = note["vlm"][1]["seconds"]
    rows.append(row)
times = pd.DataFrame(rows).set_index("nota")
RUNS_DIR.mkdir(exist_ok=True)
times.to_csv(RUNS_DIR / "tempos_por_nota.csv")
display(times.round(1), times.mean(numeric_only=True).round(1).rename("média (s)"))

## 5. Resumo

In [ ]:
report("notebook");

In [ ]:
if "google.colab" in sys.modules:  # baixa output/ e runs/ num zip
    !cd {ROOT} && zip -qr /content/colab_resultados.zip output runs
    from google.colab import files
    files.download("/content/colab_resultados.zip")